# Stage 4 Model Rebuild — Bitcoin Proxy Equities

## tl;dr

This notebook rebuilds the 24-month execution model for six Bitcoin proxy equities. It calculates scenario-specific dilution, treasury NAV, miner production and cash flow, HPC option value, target price, stock return, and excess return versus Bitcoin. It is an execution artifact, not the Stage 5 recommendation.

## Context & Methods

The decision question is which listed Bitcoin proxy offers the strongest 24-month equity convexity relative to holding Bitcoin directly. Stage 4 produces comparable outputs for Stage 5 interpretation.

### Key Assumptions

- Valuation date: 2026-08-25; BTC anchor: $78,511.23.
- BTC scenarios: +50%, +100%, and +200% over 24 months.
- Company valuation multiples are held constant—there is no automatic multiple expansion.
- Treasury-company issuance buys BTC at the midpoint of current and terminal BTC prices.
- Miner production scales with company hashrate growth and network-difficulty growth.
- Miner cash flow funds operating cost and capex; ATM proceeds are split between BTC purchases and liquidity.
- HPC value is probability-weighted and kept separate from BTC treasury and mining value.
- Missing company guidance is replaced with explicit editable assumptions, not presented as reported fact.

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

BTC_SPOT = 78_511.23
USDJPY = 159.22
TMP_RESULT = Path("stage4_results.json")

## Data

Reported inputs come from the workbook's `Source Data Refresh` sheet. Forecast parameters below are model assumptions and are separately identified.

In [2]:
scenarios = pd.DataFrame([
    {"scenario": "BTC +50%", "btc_multiple": 1.5, "difficulty_increase": 0.25},
    {"scenario": "BTC +100%", "btc_multiple": 2.0, "difficulty_increase": 0.45},
    {"scenario": "BTC +200%", "btc_multiple": 3.0, "difficulty_increase": 0.80},
])

companies = pd.DataFrame([
    {"company":"Strategy", "ticker":"MSTR", "type":"treasury", "price":126.83, "shares":445.559, "btc":840447, "cash":6690.0, "debt":6754.0, "preferred":15239.0,
     "dilution_rates":[0.12,0.18,0.28], "claim_growth_rates":[0.05,0.10,0.15]},
    {"company":"Metaplanet", "ticker":"3350.T", "type":"treasury", "price":330/USDJPY, "shares":1631.5442, "btc":43000, "cash":0.0, "debt":464.07, "preferred":148.14,
     "dilution_rates":[0.20,0.35,0.55], "claim_growth_rates":[0.00,0.10,0.20]},
    {"company":"Twenty One Capital", "ticker":"XXI", "type":"convert_treasury", "price":6.51, "shares":364.928724, "btc":43514, "cash":114.057427, "debt":486.5, "preferred":0.0,
     "dilution_rates":[0.02,0.04,0.07], "convert_shares":37.423077, "conversion_price":13.0},
    {"company":"American Bitcoin", "ticker":"ABTC", "type":"miner", "price":9.24, "shares":72.825077, "btc":8002, "cash":18.354, "debt":500.622, "preferred":0.0,
     "quarterly_btc":932, "cash_cost_btc":36500, "annual_capex":100.0, "hashrate_factors":[1.20,1.35,1.60], "hold_rate":0.60, "mining_multiple":5.0,
     "atm_capacity":1714.8, "atm_use_rates":[0.20,0.35,0.50], "atm_btc_share":0.50, "stock_comp_rates":[0.02,0.03,0.05], "efficiency_gain":0.10,
     "hpc_mw":0.0, "hpc_noi_per_mw":0.0, "hpc_multiple":0.0, "hpc_probability":0.0},
    {"company":"CleanSpark", "ticker":"CLSK", "type":"miner", "price":12.71, "shares":319.212047, "btc":13931, "cash":202.601, "debt":1770.878, "preferred":0.0,
     "quarterly_btc":1758, "cash_cost_btc":44317, "annual_capex":151.0, "hashrate_factors":[1.15,1.30,1.50], "hold_rate":0.40, "mining_multiple":5.0,
     "atm_capacity":0.0, "atm_use_rates":[0.00,0.00,0.00], "atm_btc_share":0.0, "stock_comp_rates":[0.05,0.08,0.12], "efficiency_gain":0.10,
     "hpc_mw":175.0, "hpc_noi_per_mw":1.50, "hpc_multiple":10.0, "hpc_probability":0.50},
    {"company":"MARA", "ticker":"MARA", "type":"miner", "price":11.83, "shares":468.991144, "btc":35577, "cash":421.3, "debt":2447.201, "preferred":0.0,
     "quarterly_btc":2422, "cash_cost_btc":38690, "annual_capex":188.606, "hashrate_factors":[1.10,1.25,1.45], "hold_rate":0.20, "mining_multiple":5.0,
     "atm_capacity":1500.0, "atm_use_rates":[0.20,0.35,0.50], "atm_btc_share":0.25, "stock_comp_rates":[0.03,0.05,0.08], "efficiency_gain":0.08,
     "hpc_mw":2000.0, "hpc_noi_per_mw":0.30, "hpc_multiple":1.0, "hpc_probability":0.25},
])

companies[["company","ticker","type","price","shares","btc","cash","debt","preferred"]]

,company,ticker,type,price,shares,btc,cash,debt,preferred
0,Strategy,MSTR,treasury,126.830000,445.559000,840447,6690.000000,6754.000,15239.00
1,Metaplanet,3350.T,treasury,2.072604,1631.544200,43000,0.000000,464.070,148.14
2,Twenty One Capital,XXI,convert_treasury,6.510000,364.928724,43514,114.057427,486.500,0.00
3,American Bitcoin,ABTC,miner,9.240000,72.825077,8002,18.354000,500.622,0.00
4,CleanSpark,CLSK,miner,12.710000,319.212047,13931,202.601000,1770.878,0.00
5,MARA,MARA,miner,11.830000,468.991144,35577,421.300000,2447.201,0.00


## Results

### 1. Calculate scenario mechanics

In [3]:
def current_nav_multiple(row):
    nav = row.btc * BTC_SPOT / 1_000_000 + row.cash - row.debt - row.preferred
    if row.type == "convert_treasury":
        market_cap = row.price * (346.807836)
    else:
        market_cap = row.price * row.shares
    return market_cap / nav


def treasury_case(row, scenario, index):
    terminal_btc_price = BTC_SPOT * scenario.btc_multiple
    average_btc_price = (BTC_SPOT + terminal_btc_price) / 2
    valuation_multiple = current_nav_multiple(row)
    dilution_rate = row.dilution_rates[index]
    average_issue_price = row.price * math.sqrt(scenario.btc_multiple)
    new_shares = row.shares * dilution_rate
    equity_proceeds = new_shares * average_issue_price
    claim_growth = row.claim_growth_rates[index]
    new_debt = row.debt * claim_growth
    new_preferred = row.preferred * claim_growth
    btc_purchased = (equity_proceeds + new_debt + new_preferred) * 1_000_000 / average_btc_price
    ending_btc = row.btc + btc_purchased
    ending_shares = row.shares + new_shares
    ending_debt = row.debt + new_debt
    ending_preferred = row.preferred + new_preferred
    common_nav = ending_btc * terminal_btc_price / 1_000_000 + row.cash - ending_debt - ending_preferred
    equity_value = max(common_nav * valuation_multiple, 0)
    target_price = equity_value / ending_shares
    return dict(ending_shares=ending_shares, ending_btc=ending_btc, production_24m=0.0,
                annual_ebitda=0.0, treasury_value=ending_btc*terminal_btc_price/1_000_000,
                mining_ev=0.0, hpc_value=0.0, net_debt_pref=ending_debt+ending_preferred-row.cash,
                equity_value=equity_value, target_price=target_price, valuation_multiple=valuation_multiple,
                financing_proceeds=equity_proceeds+new_debt+new_preferred)


def convert_treasury_case(row, scenario, index):
    terminal_btc_price = BTC_SPOT * scenario.btc_multiple
    valuation_multiple = current_nav_multiple(row)
    ending_shares_ex_convert = row.shares * (1 + row.dilution_rates[index])
    nav_debt_retained = row.btc * terminal_btc_price / 1_000_000 + row.cash - row.debt
    preconvert_equity = max(nav_debt_retained * valuation_multiple, 0)
    preconvert_price = preconvert_equity / ending_shares_ex_convert
    converted = preconvert_price >= row.conversion_price
    ending_shares = ending_shares_ex_convert + (row.convert_shares if converted else 0)
    ending_debt = 0.0 if converted else row.debt
    common_nav = row.btc * terminal_btc_price / 1_000_000 + row.cash - ending_debt
    equity_value = max(common_nav * valuation_multiple, 0)
    target_price = equity_value / ending_shares
    return dict(ending_shares=ending_shares, ending_btc=row.btc, production_24m=0.0,
                annual_ebitda=0.0, treasury_value=row.btc*terminal_btc_price/1_000_000,
                mining_ev=0.0, hpc_value=0.0, net_debt_pref=ending_debt-row.cash,
                equity_value=equity_value, target_price=target_price, valuation_multiple=valuation_multiple,
                financing_proceeds=0.0, converted=converted)


def miner_case(row, scenario, index):
    terminal_btc_price = BTC_SPOT * scenario.btc_multiple
    average_btc_price = (BTC_SPOT + terminal_btc_price) / 2
    difficulty_terminal = 1 + scenario.difficulty_increase
    difficulty_average = 1 + scenario.difficulty_increase / 2
    hashrate_terminal = row.hashrate_factors[index]
    hashrate_average = (1 + hashrate_terminal) / 2
    production_24m = 8 * row.quarterly_btc * hashrate_average / difficulty_average
    annual_production_terminal = 4 * row.quarterly_btc * hashrate_terminal / difficulty_terminal
    terminal_cash_cost = row.cash_cost_btc * difficulty_terminal / (1 + row.efficiency_gain)
    average_cash_cost = (row.cash_cost_btc + terminal_cash_cost) / 2
    annual_ebitda = annual_production_terminal * max(terminal_btc_price - terminal_cash_cost, 0) / 1_000_000
    mining_ev = annual_ebitda * row.mining_multiple
    atm_proceeds = row.atm_capacity * row.atm_use_rates[index]
    average_issue_price = row.price * math.sqrt(scenario.btc_multiple)
    atm_shares = atm_proceeds / average_issue_price if average_issue_price else 0
    stock_comp_shares = row.shares * row.stock_comp_rates[index]
    ending_shares = row.shares + atm_shares + stock_comp_shares
    produced_btc_held = production_24m * row.hold_rate
    atm_btc_purchases = atm_proceeds * row.atm_btc_share * 1_000_000 / average_btc_price
    ending_btc = row.btc + produced_btc_held + atm_btc_purchases
    sold_btc = production_24m * (1 - row.hold_rate)
    sold_btc_cash = sold_btc * average_btc_price / 1_000_000
    operating_cash_cost = production_24m * average_cash_cost / 1_000_000
    capex_24m = row.annual_capex * 2
    retained_atm_cash = atm_proceeds * (1 - row.atm_btc_share)
    free_cash_flow = sold_btc_cash - operating_cash_cost - capex_24m + retained_atm_cash
    net_debt = row.debt - row.cash - free_cash_flow
    hpc_value = row.hpc_mw * row.hpc_noi_per_mw * row.hpc_multiple * row.hpc_probability
    treasury_value = ending_btc * terminal_btc_price / 1_000_000
    equity_value = max(treasury_value + mining_ev + hpc_value - net_debt - row.preferred, 0)
    target_price = equity_value / ending_shares
    return dict(ending_shares=ending_shares, ending_btc=ending_btc, production_24m=production_24m,
                annual_ebitda=annual_ebitda, treasury_value=treasury_value, mining_ev=mining_ev,
                hpc_value=hpc_value, net_debt_pref=net_debt+row.preferred, equity_value=equity_value,
                target_price=target_price, valuation_multiple=row.mining_multiple,
                financing_proceeds=atm_proceeds)


records = []
for _, company in companies.iterrows():
    for idx, scenario in scenarios.iterrows():
        if company.type == "treasury":
            result = treasury_case(company, scenario, idx)
        elif company.type == "convert_treasury":
            result = convert_treasury_case(company, scenario, idx)
        else:
            result = miner_case(company, scenario, idx)
        stock_return = result["target_price"] / company.price - 1
        records.append({
            "company": company.company,
            "ticker": company.ticker,
            "type": company.type,
            "scenario": scenario.scenario,
            "btc_multiple": scenario.btc_multiple,
            "btc_terminal": BTC_SPOT * scenario.btc_multiple,
            "btc_return": scenario.btc_multiple - 1,
            **result,
            "stock_return": stock_return,
            "excess_return": stock_return - (scenario.btc_multiple - 1),
        })

results = pd.DataFrame(records)
results.round(3)

,company,ticker,type,scenario,btc_multiple,btc_terminal,btc_return,ending_shares,ending_btc,production_24m,annual_ebitda,treasury_value,mining_ev,hpc_value,net_debt_pref,equity_value,target_price,valuation_multiple,financing_proceeds,stock_return,excess_return,converted
0,Strategy,MSTR,treasury,BTC +50%,1.5,117766.845,0.5,499.026,936279.674,0.000,0.000,110262.703,0.000,0.0,16402.650,104654.597,209.718,1.115,9404.926,0.654,0.154,NaN
1,Strategy,MSTR,treasury,BTC +100%,2.0,157022.460,1.0,525.760,981271.530,0.000,0.000,154081.670,0.000,0.0,17502.300,152286.926,289.651,1.115,16584.461,1.284,0.284,NaN
2,Strategy,MSTR,treasury,BTC +200%,3.0,235533.690,2.0,570.316,1035992.044,0.000,0.000,244011.029,0.000,0.0,18601.950,251332.655,440.691,1.115,30704.964,2.475,0.475,NaN
3,Metaplanet,3350.T,treasury,BTC +50%,1.5,117766.845,0.5,1957.853,51440.127,0.000,0.000,6057.941,0.000,0.0,612.210,6662.988,3.403,1.224,828.306,0.642,0.142,NaN
4,Metaplanet,3350.T,treasury,BTC +100%,2.0,157022.460,1.0,2202.585,57732.503,0.000,0.000,9065.300,0.000,0.0,673.431,10267.660,4.662,1.224,1735.000,1.249,0.249,NaN
5,Metaplanet,3350.T,treasury,BTC +200%,3.0,235533.690,2.0,2528.894,64295.019,0.000,0.000,15143.643,0.000,0.0,734.652,17629.759,6.971,1.224,3343.796,2.364,0.364,NaN
6,Twenty One Capital,XXI,convert_treasury,BTC +50%,1.5,117766.845,0.5,372.227,43514.000,0.000,0.000,5124.506,0.000,0.0,372.443,3524.703,9.469,0.742,0.000,0.455,-0.045,False
7,Twenty One Capital,XXI,convert_treasury,BTC +100%,2.0,157022.460,1.0,379.526,43514.000,0.000,0.000,6832.675,0.000,0.0,372.443,4791.686,12.625,0.742,0.000,0.939,-0.061,False
8,Twenty One Capital,XXI,convert_treasury,BTC +200%,3.0,235533.690,2.0,427.897,43514.000,0.000,0.000,10249.013,0.000,0.0,-114.057,7686.500,17.963,0.742,0.000,1.759,-0.241,True
9,American Bitcoin,ABTC,miner,BTC +50%,1.5,117766.845,0.5,104.587,14123.504,7290.311,273.031,1663.280,1365.156,0.0,508.842,2519.595,24.091,5.000,342.960,1.607,1.107,NaN


### 2. Compare modeled returns with Bitcoin

In [4]:
return_table = results.pivot(index=["company","ticker"], columns="scenario", values=["target_price","stock_return","excess_return"])
return_table.round(3)

target_price                    stock_return                    excess_return                   
scenario                     BTC +100% BTC +200% BTC +50%    BTC +100% BTC +200% BTC +50%     BTC +100% BTC +200% BTC +50%
company            ticker                                                                                                 
American Bitcoin   ABTC         32.021    48.195   24.091        2.465     4.216    1.607         1.465     2.216    1.107
CleanSpark         CLSK         16.888    25.544   12.088        0.329     1.010   -0.049        -0.671    -0.990   -0.549
MARA               MARA         18.554    28.905   12.811        0.568     1.443    0.083        -0.432    -0.557   -0.417
Metaplanet         3350.T        4.662     6.971    3.403        1.249     2.364    0.642         0.249     0.364    0.142
Strategy           MSTR        289.651   440.691  209.718        1.284     2.475    0.654         0.284     0.475    0.154
Twenty One Capital XXI          12.625    17.963    9.469        0.939     1.759    0.455        -0.061    -0.241   -0.045

### 3. Validate model identities and bounds

In [5]:
checks = {
    "18 company-scenario rows": len(results) == 18,
    "No negative equity values": bool((results.equity_value >= 0).all()),
    "No missing target prices": bool(results.target_price.notna().all()),
    "Ending shares exceed zero": bool((results.ending_shares > 0).all()),
    "Ending BTC at least starting BTC": bool((results.ending_btc > 0).all()),
    "Returns reconcile": bool(np.allclose(results.target_price / results.company.map(companies.set_index("company").price) - 1, results.stock_return)),
}
pd.Series(checks, name="passed")

18 company-scenario rows            True
No negative equity values           True
No missing target prices            True
Ending shares exceed zero           True
Ending BTC at least starting BTC    True
Returns reconcile                   True
Name: passed, dtype: bool

In [6]:
summary = results[["company","ticker","scenario","btc_terminal","ending_shares","ending_btc","production_24m","annual_ebitda","treasury_value","mining_ev","hpc_value","net_debt_pref","equity_value","target_price","stock_return","excess_return"]].copy()
TMP_RESULT.write_text(json.dumps({"results": summary.to_dict(orient="records"), "checks": checks}, indent=2))
summary.round(3)

,company,ticker,scenario,btc_terminal,ending_shares,ending_btc,production_24m,annual_ebitda,treasury_value,mining_ev,hpc_value,net_debt_pref,equity_value,target_price,stock_return,excess_return
0,Strategy,MSTR,BTC +50%,117766.845,499.026,936279.674,0.000,0.000,110262.703,0.000,0.0,16402.650,104654.597,209.718,0.654,0.154
1,Strategy,MSTR,BTC +100%,157022.460,525.760,981271.530,0.000,0.000,154081.670,0.000,0.0,17502.300,152286.926,289.651,1.284,0.284
2,Strategy,MSTR,BTC +200%,235533.690,570.316,1035992.044,0.000,0.000,244011.029,0.000,0.0,18601.950,251332.655,440.691,2.475,0.475
3,Metaplanet,3350.T,BTC +50%,117766.845,1957.853,51440.127,0.000,0.000,6057.941,0.000,0.0,612.210,6662.988,3.403,0.642,0.142
4,Metaplanet,3350.T,BTC +100%,157022.460,2202.585,57732.503,0.000,0.000,9065.300,0.000,0.0,673.431,10267.660,4.662,1.249,0.249
5,Metaplanet,3350.T,BTC +200%,235533.690,2528.894,64295.019,0.000,0.000,15143.643,0.000,0.0,734.652,17629.759,6.971,2.364,0.364
6,Twenty One Capital,XXI,BTC +50%,117766.845,372.227,43514.000,0.000,0.000,5124.506,0.000,0.0,372.443,3524.703,9.469,0.455,-0.045
7,Twenty One Capital,XXI,BTC +100%,157022.460,379.526,43514.000,0.000,0.000,6832.675,0.000,0.0,372.443,4791.686,12.625,0.939,-0.061
8,Twenty One Capital,XXI,BTC +200%,235533.690,427.897,43514.000,0.000,0.000,10249.013,0.000,0.0,-114.057,7686.500,17.963,1.759,-0.241
9,American Bitcoin,ABTC,BTC +50%,117766.845,104.587,14123.504,7290.311,273.031,1663.280,1365.156,0.0,508.842,2519.595,24.091,1.607,1.107


## Takeaways

- The workbook should treat scenario outputs as model results, not forecasts supplied by management.
- Treasury-company returns are driven by BTC exposure, capital-structure leverage, issuance price, and dilution.
- Miner returns are driven by BTC price, difficulty, hashrate growth, cash cost, capex, treasury retention, and ATM dilution.
- HPC option value is separately visible and can be removed without changing the BTC/mining calculations.
- Stage 5 should interpret robustness across scenarios and explicitly test the assumptions with the greatest valuation impact.